[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Object-Oriented Python](https://johnfisher-ai.github.io/Python-Visual-Guides/object-oriented-python.html)

# Context Managers and Iterators &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.


**1.** A context manager that marks where a section starts and ends.


In [1]:
class Section:
    def __init__(self, title):
        self.title = title

    def __enter__(self):
        print(f"== {self.title} ==")
        return self

    def __exit__(self, exc_type, exc, tb):
        print(f"== end {self.title} ==")
        return False


with Section("north"):
    print("Tromso -4.1")
    print("Bodo -2.6")


== north ==
Tromso -4.1
Bodo -2.6
== end north ==


`__enter__` returns `self` by convention even though nothing here uses `as`. It costs nothing, and
it means `with Section("north") as section:` would work if somebody needed it later.


**2.** The closing line prints even when the block raises.


In [2]:
try:
    with Section("south"):
        print("Malaga 18.9")
        raise ValueError("instrument fault")
        print("never reached")
except ValueError as error:
    print("caught outside the with:", error)


== south ==
Malaga 18.9
== end south ==
caught outside the with: instrument fault


The order is the thing to read. The block raised, `__exit__` ran and printed its line, and only then
did the exception reach the `except`. Returning `False` from `__exit__` is what let it continue on
to the `except`; returning `True` would have hidden it.


**3.** A temporary change, restored on every way out.


In [3]:
class Station:
    def __init__(self, name, readings, unit="C"):
        self.name = name
        self.readings = readings
        self.unit = unit


class TemporaryUnit:
    def __init__(self, station, unit):
        self.station = station
        self.unit = unit

    def __enter__(self):
        self.previous = self.station.unit
        self.station.unit = self.unit
        return self.station

    def __exit__(self, exc_type, exc, tb):
        self.station.unit = self.previous
        return False


north = Station("Tromso", [-4.1, -2.6])

with TemporaryUnit(north, "F") as station:
    print("inside: ", station.unit)
print("after:  ", north.unit)

try:
    with TemporaryUnit(north, "K"):
        print("inside: ", north.unit)
        raise RuntimeError("stopped halfway")
except RuntimeError:
    pass
print("after an error:", north.unit)


inside:  F
after:   C
inside:  K
after an error: C


The previous value is saved in `__enter__`, not in `__init__`. That matters if the same
`TemporaryUnit` is used twice: saving at creation would restore whatever the unit was when the
object was built, which may no longer be the value to go back to.

`__enter__` returns the station rather than `self`, because the station is what the block wants to
work with. `as` binds whatever `__enter__` returns.


**4.** An iterable written with the explicit protocol.


In [4]:
class ShelfIterator:
    def __init__(self, titles):
        self.titles = titles
        self.position = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.position >= len(self.titles):
            raise StopIteration
        title = self.titles[self.position]
        self.position += 1
        return title


class Shelf:
    def __init__(self, titles):
        self.titles = titles

    def __iter__(self):
        return ShelfIterator(self.titles)


shelf = Shelf(["Dune", "Emma", "Ethan Frome"])

for title in shelf:
    print(title)
print("again:", list(shelf))


Dune
Emma
Ethan Frome
again: ['Dune', 'Emma', 'Ethan Frome']


The second loop works because `Shelf.__iter__` builds a new `ShelfIterator` each time, starting at
position zero. The position lives on the iterator, not on the shelf, so two loops never share it.


**5.** The same shelf, with `yield`.


In [5]:
class Shelf:
    def __init__(self, titles):
        self.titles = titles

    def __iter__(self):
        for title in self.titles:
            yield title


shelf = Shelf(["Dune", "Emma", "Ethan Frome"])

print("list:     ", list(shelf))
print("how many: ", len(list(shelf)))
print("has Emma: ", "Emma" in shelf)
print("again:    ", list(shelf))


list:      ['Dune', 'Emma', 'Ethan Frome']
how many:  3
has Emma:  True
again:     ['Dune', 'Emma', 'Ethan Frome']


`ShelfIterator` is gone entirely. Each call to `__iter__` returns a new generator, and the generator
remembers its position by remembering where the function paused, so there is nothing to track by
hand.

`len(shelf)` would still fail, because iteration and length are separate protocols. Add `__len__`
from the **Dunder Methods** notebook if a shelf should have a length.


**6.** A countdown that yields its values.


In [6]:
class Countdown:
    def __init__(self, start):
        self.start = start

    def __iter__(self):
        value = self.start
        while value > 0:
            yield value
            value -= 1


for number in Countdown(3):
    print(number)
print("sum of Countdown(4):", sum(Countdown(4)))


3
2
1
sum of Countdown(4): 10


Nothing stores the numbers. Each value is produced when the loop asks for it, and discarded once
the loop moves on, which is why a `Countdown(1_000_000)` would take no more memory than this one.


---

&#8592; **Back to:** [Context Managers and Iterators](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/object-oriented-python/05-context-managers-and-iterators.ipynb)  &nbsp;&middot;&nbsp;  [Object-Oriented Python Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/object-oriented-python.html)
